In [ ]:
import pandas as pd

# Load the data
# file_path = "/Users/doughnut/Library/CloudStorage/OneDrive-TheUniversityofMelbourne/Phase 1 - Australian Genetic and Genomic Test Utilisation/Data Files/Processed_MBSGeneticsCount_20240325_175333.feather"
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/Data Files/Processed_MBSGeneticsCount_20240325_175333.feather"
data = pd.read_feather(file_path)

data.info()


#### Logistic Curve - All

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# Load the data (using your file path)
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/Data Files/Processed_MBSGeneticsCount_20240325_175333.feather"
data = pd.read_feather(file_path)

# Ensure the data is sorted by month
data = data.sort_values(by='Month')

# Aggregating by month
monthly_data = data.groupby('Month')['Value'].sum().reset_index()

# Logistic function (diffusion curve)
def logistic_function(x, L, k, x0):
    """
    Logistic function for diffusion modeling.
    L : the curve's maximum value (carrying capacity)
    k : the growth rate
    x0: the x-value of the sigmoid's midpoint
    """
    return L / (1 + np.exp(-k * (x - x0)))

# Prepare the data for curve fitting
x_data = (monthly_data['Month'] - monthly_data['Month'].min()).dt.days  # Days since the first month
y_data = monthly_data['Value'].values

# Initial guess for parameters [L, k, x0]
initial_guess = [max(y_data), 1, np.median(x_data)]

# Fit the logistic function to the data
params, covariance = curve_fit(logistic_function, x_data, y_data, p0=initial_guess)

# Extract the fitted parameters
L, k, x0 = params

# Generate fitted values for the curve
x_fit = np.linspace(min(x_data), max(x_data), 100)
y_fit = logistic_function(x_fit, L, k, x0)

# Plot the results
plt.figure(figsize=(10, 6))
plt.scatter(x_data, y_data, label='Observed Data')
plt.plot(x_fit, y_fit, color='red', label='Fitted Logistic Curve')
plt.title('Diffusion Curve Fitting')
plt.xlabel('Days Since Start')
plt.ylabel('Test Utilization (Value)')
plt.legend()
plt.grid(True)
plt.show()

# Print the fitted parameters
print(f"Fitted parameters: L={L:.2f}, k={k:.4f}, x0={x0:.2f}")

# Check for diffusion pattern: Logistic curves are generally expected to have k > 0. 
if k > 0:
    print("The test follows a diffusion pattern.")
else:
    print("The test does not follow a diffusion pattern.")


#### Bass Diffusion Model - All

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# Load the data (using your file path)
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/Data Files/Processed_MBSGeneticsCount_20240325_175333.feather"
data = pd.read_feather(file_path)

# Ensure the data is sorted by month
data = data.sort_values(by='Month')

# Aggregating by month
monthly_data = data.groupby('Month')['Value'].sum().reset_index()

# Bass Diffusion Model
def bass_diffusion(t, p, q, m):
    """
    Bass diffusion model.
    t : time (days)
    p : coefficient of innovation
    q : coefficient of imitation
    m : market potential (maximum adoption level)
    """
    adoption = (p + q * (np.cumsum(np.ones(len(t))) / m)) * (m - np.cumsum(np.ones(len(t))))
    return adoption

# Prepare the data for curve fitting
x_data = (monthly_data['Month'] - monthly_data['Month'].min()).dt.days  # Days since the first month
y_data = monthly_data['Value'].values

# Initial guess for parameters [p, q, m]
initial_guess = [0.03, 0.38, max(y_data)]

# Increase maxfev to allow more iterations
maxfev_value = 2000

# Fit the Bass diffusion model to the data
params, covariance = curve_fit(bass_diffusion, x_data, y_data, p0=initial_guess, maxfev=maxfev_value)

# Extract the fitted parameters
p, q, m = params

# Generate fitted values for the curve
x_fit = np.linspace(min(x_data), max(x_data), len(x_data))
y_fit = bass_diffusion(x_fit, p, q, m)

# Plot the results
plt.figure(figsize=(10, 6))
plt.scatter(x_data, y_data, label='Observed Data')
plt.plot(x_fit, y_fit, color='red', label='Fitted Bass Diffusion Curve')
plt.title('Bass Diffusion Curve Fitting')
plt.xlabel('Days Since Start')
plt.ylabel('Test Utilization (Value)')
plt.legend()
plt.grid(True)
plt.show()

# Print the fitted parameters
print(f"Fitted parameters: p={p:.4f}, q={q:.4f}, m={m:.2f}")

# Check for diffusion pattern
if p > 0 and q > 0:
    print("The test follows a diffusion of innovation pattern.")
else:
    print("The test does not follow a diffusion of innovation pattern.")


#### Gompertz Curve - All

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# Load the data (using your file path)
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/Data Files/Processed_MBSGeneticsCount_20240325_175333.feather"
data = pd.read_feather(file_path)

# Ensure the data is sorted by month
data = data.sort_values(by='Month')

# Aggregating by month
monthly_data = data.groupby('Month')['Value'].sum().reset_index()

# Gompertz Model
def gompertz_function(t, L, k, t0):
    """
    Gompertz function.
    L : the curve's maximum value (asymptote)
    k : the growth rate
    t0: the x-value of the inflection point (time offset)
    """
    return L * np.exp(-np.exp(-k * (t - t0)))

# Prepare the data for curve fitting
x_data = (monthly_data['Month'] - monthly_data['Month'].min()).dt.days  # Days since the first month
y_data = monthly_data['Value'].values

# Initial guess for parameters [L, k, t0]
initial_guess = [max(y_data), 0.1, np.median(x_data)]

# Fit the Gompertz model to the data
params, covariance = curve_fit(gompertz_function, x_data, y_data, p0=initial_guess, maxfev=2000)

# Extract the fitted parameters
L, k, t0 = params

# Generate fitted values for the curve
x_fit = np.linspace(min(x_data), max(x_data), 100)
y_fit = gompertz_function(x_fit, L, k, t0)

# Plot the results
plt.figure(figsize=(10, 6))
plt.scatter(x_data, y_data, label='Observed Data')
plt.plot(x_fit, y_fit, color='red', label='Fitted Gompertz Curve')
plt.title('Gompertz Curve Fitting')
plt.xlabel('Days Since Start')
plt.ylabel('Test Utilization (Value)')
plt.legend()
plt.grid(True)
plt.show()

# Print the fitted parameters
print(f"Fitted parameters: L={L:.2f}, k={k:.4f}, t0={t0:.2f}")

# Check for growth pattern
if k > 0:
    print("The test follows a Gompertz growth pattern.")
else:
    print("The test does not follow a Gompertz growth pattern.")


#### Gompertz Curve - Items

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# Load the data (using your file path)
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/Data Files/Processed_MBSGeneticsCount_20240325_175333.feather"
data = pd.read_feather(file_path)

# Ensure the data is sorted by month
data = data.sort_values(by=['Item', 'Month'])

# Gompertz Model
def gompertz_function(t, L, k, t0):
    """
    Gompertz function.
    L : the curve's maximum value (asymptote)
    k : the growth rate
    t0: the x-value of the inflection point (time offset)
    """
    return L * np.exp(-np.exp(-k * (t - t0)))

# Function to fit Gompertz model for a single category and return the parameters
def fit_gompertz_to_item(item_data):
    # Prepare the data for curve fitting
    x_data = (item_data['Month'] - item_data['Month'].min()).dt.days  # Days since the first month
    y_data = item_data['Value'].values

    # Initial guess for parameters [L, k, t0]
    initial_guess = [max(y_data), 0.1, np.median(x_data)]

    try:
        # Fit the Gompertz model to the data
        params, _ = curve_fit(gompertz_function, x_data, y_data, p0=initial_guess, maxfev=2000)
        return params
    except RuntimeError:
        print(f"Could not fit Gompertz model for Item {item_data['Item'].iloc[0]}")
        return None

# Dictionary to store results for each category
gompertz_params = {}

# Loop over each category in "Item"
for item, item_data in data.groupby('Item'):
    print(f"Fitting Gompertz model for Item: {item}")
    params = fit_gompertz_to_item(item_data)
    if params is not None:
        gompertz_params[item] = params
        L, k, t0 = params

        # Generate fitted values for the curve
        x_data = (item_data['Month'] - item_data['Month'].min()).dt.days
        x_fit = np.linspace(min(x_data), max(x_data), 100)
        y_fit = gompertz_function(x_fit, L, k, t0)

        # Plot the results
        plt.figure(figsize=(10, 6))
        plt.scatter(x_data, item_data['Value'], label='Observed Data')
        plt.plot(x_fit, y_fit, color='red', label=f'Fitted Gompertz Curve for Item {item}')
        plt.title(f'Gompertz Curve Fitting for Item {item}')
        plt.xlabel('Days Since Start')
        plt.ylabel('Test Utilization (Value)')
        plt.legend()
        plt.grid(True)
        plt.show()

        # Print the fitted parameters
        print(f"Item {item} - Fitted parameters: L={L:.2f}, k={k:.4f}, t0={t0:.2f}\n")

# gompertz_params now contains the fitted parameters for each item
print("Fitted parameters for each item:")
for item, params in gompertz_params.items():
    print(f"Item {item}: L={params[0]:.2f}, k={params[1]:.4f}, t0={params[2]:.2f}")


#### Gompertz Curve

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# Load the data (using your file path)
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/Data Files/Processed_MBSGeneticsCount_20240325_175333.feather"
data = pd.read_feather(file_path)

# Ensure the data is sorted by month
data = data.sort_values(by=['Item', 'Month'])

# Gompertz Model
def gompertz_function(t, L, k, t0):
    """
    Gompertz function.
    L : the curve's maximum value (asymptote)
    k : the growth rate
    t0: the x-value of the inflection point (time offset)
    """
    return L * np.exp(-np.exp(-k * (t - t0)))

# Function to fit Gompertz model for a single category and return the parameters
def fit_gompertz_to_item(item_data):
    # Prepare the data for curve fitting
    x_data = (item_data['Month'] - item_data['Month'].min()).dt.days  # Days since the first month
    y_data = item_data['Value'].values

    # Initial guess for parameters [L, k, t0]
    initial_guess = [max(y_data), 0.1, np.median(x_data)]

    try:
        # Fit the Gompertz model to the data
        params, _ = curve_fit(gompertz_function, x_data, y_data, p0=initial_guess, maxfev=2000)
        return params
    except RuntimeError:
        print(f"Could not fit Gompertz model for Item {item_data['Item'].iloc[0]}")
        return None

# Lists to store results
gompertz_followers = []
gompertz_non_followers = []

# Loop over each category in "Item"
for item, item_data in data.groupby('Item'):
    print(f"Fitting Gompertz model for Item: {item}")
    params = fit_gompertz_to_item(item_data)
    if params is not None:
        L, k, t0 = params

        # Generate fitted values for the curve
        x_data = (item_data['Month'] - item_data['Month'].min()).dt.days
        x_fit = np.linspace(min(x_data), max(x_data), 100)
        y_fit = gompertz_function(x_fit, L, k, t0)

        # Plot the results
        plt.figure(figsize=(10, 6))
        plt.scatter(x_data, item_data['Value'], label='Observed Data')
        plt.plot(x_fit, y_fit, color='red', label=f'Fitted Gompertz Curve for Item {item}')
        plt.title(f'Gompertz Curve Fitting for Item {item}')
        plt.xlabel('Days Since Start')
        plt.ylabel('Test Utilization (Value)')
        plt.legend()
        plt.grid(True)
        plt.show()

        # Print the fitted parameters
        print(f"Item {item} - Fitted parameters: L={L:.2f}, k={k:.4f}, t0={t0:.2f}\n")

        # Check for Gompertz growth pattern (if k > 0)
        if k > 0:
            gompertz_followers.append({'Item': item, 'L': L, 'k': k, 't0': t0})
            print(f"Item {item} follows a Gompertz growth pattern.\n")
        else:
            gompertz_non_followers.append({'Item': item, 'L': L, 'k': k, 't0': t0})
            print(f"Item {item} does not follow a Gompertz growth pattern.\n")
    else:
        gompertz_non_followers.append({'Item': item, 'L': None, 'k': None, 't0': None})
        print(f"Item {item} could not be fitted with a Gompertz model.\n")

# Convert results to DataFrames for summary
gompertz_followers_df = pd.DataFrame(gompertz_followers)
gompertz_non_followers_df = pd.DataFrame(gompertz_non_followers)

# Display the summaries
print("Items that follow a Gompertz growth pattern:")
print(gompertz_followers_df)

print("\nItems that do not follow a Gompertz growth pattern or could not be fitted:")
print(gompertz_non_followers_df)


#### Neural Prophet - All

In [ ]:
import pandas as pd
from neuralprophet import NeuralProphet
import matplotlib.pyplot as plt

# Load the data (using your file path)
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/Data Files/Processed_MBSGeneticsCount_20240325_175333.feather"
data = pd.read_feather(file_path)

# Ensure the data is sorted by month and aggregate by 'Month'
data = data.groupby('Month', as_index=False)['Value'].sum()

# Prepare the data for NeuralProphet
data.rename(columns={'Month': 'ds', 'Value': 'y'}, inplace=True)

# Initialize the NeuralProphet model
model = NeuralProphet()

# Fit the model to the aggregated data
model.fit(data, freq='M')

# Make future predictions
future = model.make_future_dataframe(data, periods=12)  # Predict the next 12 months
forecast = model.predict(future)

# Use matplotlib for static plots
fig_forecast = model.plot(forecast)
plt.show()

# Plot components like trend, seasonality, etc.
fig_components = model.plot_components(forecast)
plt.show()


In [ ]:
import pandas as pd
from econml.dr import DRLearner
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the data
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/Data Files/Processed_MBSGeneticsCount_20240325_175333.feather"
data = pd.read_feather(file_path)

# Data Overview
print(data.info())

# Ensure 'Month' is the datetime index and aggregate by 'Item' and 'Month'
data.set_index('Month', inplace=True)
data_grouped = data.groupby(['Item', pd.Grouper(freq='ME')])['Value'].sum().reset_index()

# Convert categorical variables to dummy/one-hot encoded features
data_grouped = pd.get_dummies(data_grouped, columns=['Item'], drop_first=True)

# Define treatment and control groups (this is a placeholder, modify based on your criteria)
# For example, you can define a threshold on "Value" to assign treatment.
data_grouped['treatment'] = (data_grouped['Value'] > 1000).astype(int)  # Example treatment definition

# Define features and target
X = data_grouped.drop(columns=['Value', 'Month', 'treatment'])
y = data_grouped['Value']
T = data_grouped['treatment']  # Binary treatment variable

# Standardize features to avoid issues with scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split the data for training and testing
X_train, X_test, y_train, y_test, T_train, T_test = train_test_split(X_scaled, y, T, test_size=0.3, random_state=42)

# Define the uplift model using DRLearner
base_learner = RandomForestRegressor(n_estimators=100, min_samples_leaf=10)
propensity_model = LogisticRegression()  # Use Logistic Regression for propensity

dr_learner = DRLearner(model_propensity=propensity_model,
                       model_regression=base_learner)

# Fit the model
dr_learner.fit(Y=y_train, T=T_train, X=X_train)

# Estimate treatment effects
treatment_effects = dr_learner.effect(X_test)

# Display treatment effects
print("Estimated Treatment Effects:", treatment_effects)

# Optionally visualize results
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.hist(treatment_effects, bins=20, color='blue', alpha=0.7)
plt.title('Distribution of Estimated Treatment Effects')
plt.xlabel('Treatment Effect')
plt.ylabel('Frequency')
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from econml.dml import LinearDML
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LassoCV

# Load the data
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/Data Files/Processed_MBSGeneticsCount_20240325_175333.feather"
data = pd.read_feather(file_path)

# Aggregate by 'Item' and 'Month', ignoring 'State'
data_agg = data.groupby(['Item', 'Month'], observed=True)['Value'].sum().reset_index()

# Calculate the monthly growth rate for each test
data_agg['GrowthRate'] = data_agg.groupby('Item', observed=True)['Value'].pct_change()

# Replace NaN or infinite values in GrowthRate
data_agg['GrowthRate'] = data_agg['GrowthRate'].replace([np.inf, -np.inf], np.nan)
data_agg['GrowthRate'] = data_agg['GrowthRate'].fillna(0)

# Define treatment and outcome
threshold_value = 500  # Define your own threshold for what constitutes "introduction"
data_agg['Treatment'] = (data_agg['Value'] > threshold_value).astype(int)

# One-hot encode the 'Item' column to create features for econml
encoder = OneHotEncoder(sparse_output=False, drop='first')
item_features = encoder.fit_transform(data_agg[['Item']])

# Define the outcome as the growth rate of other tests
outcome = data_agg['GrowthRate']

# Define features for the model (item features for now)
X = item_features
T = data_agg['Treatment']

# Define the model using a classifier for the treatment
model = LinearDML(model_y=LassoCV(), model_t=RandomForestClassifier(), discrete_treatment=True)

# Fit the model
model.fit(outcome, T, X=X)

# Estimate the treatment effect
treatment_effects = model.effect(X)

# Add the treatment effects to the dataframe
data_agg['TreatmentEffect'] = treatment_effects

# View the first few rows of the data with treatment effects
print(data_agg.head())


In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from econml.dml import LinearDML
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Load the data
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/Data Files/Processed_MBSGeneticsCount_20240325_175333.feather"
data = pd.read_feather(file_path)

# Aggregate by 'Item' and 'Month', ignoring 'State'
data = data.groupby(['Item', 'Month'], as_index=False)['Value'].sum()

# Encode the 'Item' column as integers (for modeling)
le = LabelEncoder()
data['Item_encoded'] = le.fit_transform(data['Item'])

# Create lag features to capture growth patterns
data['Value_lag1'] = data.groupby('Item')['Value'].shift(1)
data['Value_lag1'].fillna(0, inplace=True)

# Create binary treatment variable for uplift modeling
# The treatment will be whether the test had a significant increase in 'Value' (defined as >50% growth)
data['Treatment'] = (data['Value'] > data['Value_lag1'] * 1.5).astype(int)

# Create features and labels
X = data[['Item_encoded', 'Value_lag1']]
y = data['Value']
treatment = data['Treatment']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test, treatment_train, treatment_test = train_test_split(
    X, y, treatment, test_size=0.2, random_state=42)

# Define the base models for outcome and treatment
model_y = XGBRegressor(random_state=42)
model_t = XGBRegressor(random_state=42)

# Apply double machine learning using LinearDML from econml
dml = LinearDML(model_y=model_y, model_t=model_t)

# Fit the DML model to estimate the treatment effects
dml.fit(y_train, treatment_train, X=X_train)

# Estimate the treatment effect for test categories
treatment_effects = dml.effect(X_test)

# Create a DataFrame to store results
results = pd.DataFrame({
    'Item_encoded': X_test['Item_encoded'],
    'Estimated_Treatment_Effect': treatment_effects
})

# Decode 'Item' to its original form
results['Item'] = le.inverse_transform(results['Item_encoded'])

# Display the results
print(results.head())


In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from econml.dml import LinearDML
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Load the data
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/Data Files/Processed_MBSGeneticsCount_20240325_175333.feather"
data = pd.read_feather(file_path)

# Aggregate by 'Item' and 'Month', ignoring 'State'
data = data.groupby(['Item', 'Month'], as_index=False)['Value'].sum()

# Encode the 'Item' column as integers (for modeling)
le = LabelEncoder()
data['Item_encoded'] = le.fit_transform(data['Item'])

# Create lag features to capture growth patterns
data['Value_lag1'] = data.groupby('Item')['Value'].shift(1)
data['Value_lag1'].fillna(0, inplace=True)

# Create a treatment variable for uplift modeling
# The treatment captures significant increase (>50% growth) or decrease (<50% decline) in 'Value'
data['Growth_Rate'] = (data['Value'] - data['Value_lag1']) / data['Value_lag1']
data['Treatment'] = np.where(data['Growth_Rate'] > 0.05, 1, np.where(data['Growth_Rate'] < -0.05, -1, 0))

# Create features and labels
X = data[['Item_encoded', 'Value_lag1']]
y = data['Value']
treatment = data['Treatment']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test, treatment_train, treatment_test = train_test_split(
    X, y, treatment, test_size=0.2, random_state=42)

# Define the base models for outcome and treatment
model_y = XGBRegressor(random_state=42)
model_t = XGBRegressor(random_state=42)

# Apply double machine learning using LinearDML from econml
dml = LinearDML(model_y=model_y, model_t=model_t)

# Fit the DML model to estimate the treatment effects
dml.fit(y_train, treatment_train, X=X_train)

# Estimate the treatment effect for test categories
treatment_effects = dml.effect(X_test)

# Create a DataFrame to store results
results = pd.DataFrame({
    'Item_encoded': X_test['Item_encoded'],
    'Estimated_Treatment_Effect': treatment_effects
})

# Decode 'Item' to its original form
results['Item'] = le.inverse_transform(results['Item_encoded'])

# Display the results
print(results.head())


In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from econml.dml import LinearDML
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Load the data
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/Data Files/Processed_MBSGeneticsCount_20240325_175333.feather"
data = pd.read_feather(file_path)

# Encode the 'Item' column as integers (for modeling)
le = LabelEncoder()
data['Item_encoded'] = le.fit_transform(data['Item'])

# Create lag features to capture growth patterns, ensure no chaining assignment issue
data['Value_lag1'] = data.groupby('Item', observed=False)['Value'].shift(1)
data['Value_lag1'] = data['Value_lag1'].fillna(0)  # Avoid inplace=True chaining assignment

# Create a treatment indicator for growth > 50% and decline > 50%, no chaining assignment
data['Growth_Rate'] = (data['Value'] - data['Value_lag1']) / data['Value_lag1']
data['Treatment'] = np.where(data['Growth_Rate'] > 0.5, 1, np.where(data['Growth_Rate'] < -0.5, -1, 0))

# Create a list of unique 'Item' categories
unique_items = data['Item'].unique()

# Dictionary to store the results of treatment effects between items
pairwise_results = {}

# Loop through each 'Item' category and evaluate its treatment effect on others
for treated_item in unique_items:
    # Use .loc[] to avoid chaining assignment issues when creating 'Is_Treated'
    data['Is_Treated'] = 0  # Start with all 0
    data.loc[data['Item'] == treated_item, 'Is_Treated'] = 1  # Set treated items to 1

    # For each treated item, split data between the treated and other items (controls)
    for control_item in unique_items:
        if treated_item != control_item:
            # Subset data to include only the treated and control items
            subset_data = data[(data['Item'] == treated_item) | (data['Item'] == control_item)].copy()
            
            # Avoid inplace modification warnings by directly assigning values
            subset_data['Item_encoded'] = le.transform(subset_data['Item'])

            # Define features and labels
            X = subset_data[['Item_encoded', 'Value_lag1']]
            y = subset_data['Value']
            treatment = subset_data['Is_Treated']

            # Split the data into training and testing sets
            X_train, X_test, y_train, y_test, treatment_train, treatment_test = train_test_split(
                X, y, treatment, test_size=0.2, random_state=42)

            # Define the base models for outcome and treatment
            model_y = XGBRegressor(random_state=42)
            model_t = XGBRegressor(random_state=42)

            # Apply double machine learning using LinearDML from econml
            dml = LinearDML(model_y=model_y, model_t=model_t)

            # Fit the DML model to estimate the treatment effects
            dml.fit(y_train, treatment_train, X=X_train)

            # Estimate the treatment effect for control items
            treatment_effects = dml.effect(X_test)

            # Store the results for each pair of treated and control items
            pairwise_results[(treated_item, control_item)] = np.mean(treatment_effects)

# Convert results to a DataFrame for easier interpretation
results_df = pd.DataFrame(list(pairwise_results.items()), columns=['Item_Pair', 'Estimated_Treatment_Effect'])

# Print the results
print(results_df.head())


In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from econml.dml import LinearDML
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from joblib import Parallel, delayed  # For parallel processing

# Load the data
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/Data Files/Processed_MBSGeneticsCount_20240325_175333.feather"
data = pd.read_feather(file_path)

# Encode the 'Item' column as integers (for modeling)
le = LabelEncoder()
data['Item_encoded'] = le.fit_transform(data['Item'])

# Create lag features to capture growth patterns
data['Value_lag1'] = data.groupby('Item', observed=False)['Value'].shift(1)
data['Value_lag1'] = data['Value_lag1'].fillna(0)  # Avoid inplace=True chaining assignment

# Create a treatment indicator for growth > 50% and decline > 50%
data['Growth_Rate'] = (data['Value'] - data['Value_lag1']) / data['Value_lag1']
data['Treatment'] = np.where(data['Growth_Rate'] > 0.5, 1, np.where(data['Growth_Rate'] < -0.5, -1, 0))

# Create a list of unique 'Item' categories
unique_items = data['Item'].unique()

# Function to calculate the treatment effect between a pair of items
def compute_treatment_effect(treated_item, control_item):
    # Subset data to include only the treated and control items
    subset_data = data[(data['Item'] == treated_item) | (data['Item'] == control_item)].copy()

    # Use GPU acceleration for XGBoost
    subset_data['Item_encoded'] = le.transform(subset_data['Item'])
    
    # Define features and labels
    X = subset_data[['Item_encoded', 'Value_lag1']]
    y = subset_data['Value']
    treatment = np.where(subset_data['Item'] == treated_item, 1, 0)  # Is treated
    
    # Split the data into training and testing sets
    X_train, X_test, y_train, y_test, treatment_train, treatment_test = train_test_split(
        X, y, treatment, test_size=0.2, random_state=42)

    # Define the base models for outcome and treatment, use GPU acceleration
    model_y = XGBRegressor(tree_method='gpu_hist', random_state=42)  # GPU-enabled
    model_t = XGBRegressor(tree_method='gpu_hist', random_state=42)  # GPU-enabled

    # Apply double machine learning using LinearDML from econml
    dml = LinearDML(model_y=model_y, model_t=model_t)

    # Fit the DML model to estimate the treatment effects
    dml.fit(y_train, treatment_train, X=X_train)

    # Estimate the treatment effect for control items
    treatment_effects = dml.effect(X_test)

    # Return the mean treatment effect for this pair
    return treated_item, control_item, np.mean(treatment_effects)

# Use joblib to parallelize the computation across multiple cores
pairwise_results = Parallel(n_jobs=-1, backend='multiprocessing')(
    delayed(compute_treatment_effect)(treated_item, control_item)
    for treated_item in unique_items
    for control_item in unique_items
    if treated_item != control_item
)

# Convert results to a DataFrame for easier interpretation
results_df = pd.DataFrame(pairwise_results, columns=['Treated_Item', 'Control_Item', 'Estimated_Treatment_Effect'])

# Print the results
print(results_df.head())


#### Neural Prophet - Model

In [ ]:
import pandas as pd
from neuralprophet import NeuralProphet
import plotly.io as pio
from IPython.display import IFrame

# Load the data
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/Data Files/Processed_MBSGeneticsCount_20240325_175333.feather"
data = pd.read_feather(file_path)

# Aggregating data by "Month"
monthly_data = data.groupby('Month', as_index=False)['Value'].sum()

# Renaming columns to fit NeuralProphet's requirements
monthly_data.rename(columns={'Month': 'ds', 'Value': 'y'}, inplace=True)

# Initialize NeuralProphet model
model = NeuralProphet()

# Fit the model on historical data
metrics = model.fit(monthly_data, freq='M')

# Make future predictions (12 future periods, in this case 12 months)
future = model.make_future_dataframe(df=monthly_data, periods=12)
forecast = model.predict(future)

# Plot historical data along with the forecast
fig = model.plot(forecast)

# Save the plot to an HTML file
html_file = 'forecast_with_historical_plot.html'
pio.write_html(fig, file=html_file, auto_open=False)

# Display the plot using an iframe
IFrame(src=html_file, width="100%", height="600px")


#### CDF

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the data
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/Data Files/Processed_MBSGeneticsCount_20240325_175333.feather"
data = pd.read_feather(file_path)

# Aggregate by 'Month' (sum values for each month)
aggregated_data = data.groupby('Month')['Value'].sum().reset_index()

# Sort data by 'Month' just in case it is not sorted
aggregated_data = aggregated_data.sort_values('Month')

# Calculate the Cumulative Density Function (CDF)
aggregated_data['CDF'] = aggregated_data['Value'].cumsum() / aggregated_data['Value'].sum()

# Plot the CDF
plt.figure(figsize=(10, 6))
plt.plot(aggregated_data['Month'], aggregated_data['CDF'], marker='o', linestyle='-', color='b')
plt.title('Cumulative Density Function of Test Utilisation Over Time')
plt.xlabel('Month')
plt.ylabel('CDF')
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the data
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/Data Files/Processed_MBSGeneticsCount_20240325_175333.feather"
data = pd.read_feather(file_path)

# Aggregate by 'Item' and 'Month' (sum values for each item and month)
aggregated_data = data.groupby(['Item', 'Month'])['Value'].sum().reset_index()

# Function to calculate and plot CDF for each category
def plot_cdf_per_item(aggregated_data):
    items = aggregated_data['Item'].unique()
    plt.figure(figsize=(10, 6))

    for item in items:
        # Filter data for the current item
        item_data = aggregated_data[aggregated_data['Item'] == item].sort_values('Month')

        # Calculate the Cumulative Density Function (CDF) for this item
        item_data['CDF'] = item_data['Value'].cumsum() / item_data['Value'].sum()

        # Plot the CDF for the current item
        plt.plot(item_data['Month'], item_data['CDF'], marker='o', linestyle='-', label=f'Item {item}')

    # Add title, labels, and grid
    plt.title('Cumulative Density Function of Test Utilisation for Each Item Over Time')
    plt.xlabel('Month')
    plt.ylabel('CDF')
    plt.grid(True)
    plt.legend(loc='upper left', bbox_to_anchor=(1, 1))  # Legend outside the plot for better readability
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# Call the function to plot CDF for each 'Item'
plot_cdf_per_item(aggregated_data)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the data
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/Data Files/Processed_MBSGeneticsCount_20240325_175333.feather"
data = pd.read_feather(file_path)

# Aggregate by 'Item' and 'Month' (sum values for each item and month)
aggregated_data = data.groupby(['Item', 'Month'])['Value'].sum().reset_index()

# Function to calculate and plot CDF for each category in separate subplots
def plot_cdf_per_item(aggregated_data):
    items = aggregated_data['Item'].unique()
    n_items = len(items)
    n_cols = 5  # Set the number of columns
    n_rows = (n_items + n_cols - 1) // n_cols  # Calculate the number of rows needed

    # Create subplots
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5 * n_rows), constrained_layout=True)
    axes = axes.flatten()  # Flatten the axes array for easier indexing

    for i, item in enumerate(items):
        # Filter data for the current item
        item_data = aggregated_data[aggregated_data['Item'] == item].sort_values('Month')

        # Calculate the Cumulative Density Function (CDF) for this item
        item_data['CDF'] = item_data['Value'].cumsum() / item_data['Value'].sum()

        # Plot the CDF for the current item in its respective subplot
        axes[i].plot(item_data['Month'], item_data['CDF'], marker='o', linestyle='-', label=f'Item {item}')
        axes[i].set_title(f'Item {item}')
        axes[i].set_xlabel('Month')
        axes[i].set_ylabel('CDF')
        axes[i].grid(True)
        axes[i].tick_params(axis='x', rotation=45)

    # Remove empty subplots (if any)
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.suptitle('Cumulative Density Function of Test Utilisation Over Time by Item', fontsize=16)
    plt.show()

# Call the function to plot CDF for each 'Item' in separate subplots
plot_cdf_per_item(aggregated_data)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the data
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/Data Files/Processed_MBSGeneticsCount_20240325_175333.feather"
data = pd.read_feather(file_path)

# Aggregate by 'Item' and 'Month' (sum values for each item and month)
aggregated_data = data.groupby(['Item', 'Month'])['Value'].sum().reset_index()

# Function to calculate and plot CDF for each category in separate subplots
def plot_cdf_per_item(aggregated_data):
    items = aggregated_data['Item'].unique()
    n_items = len(items)
    n_cols = 5  # Set the number of columns
    n_rows = (n_items + n_cols - 1) // n_cols  # Calculate the number of rows needed

    # Create subplots
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5 * n_rows), constrained_layout=True)
    axes = axes.flatten()  # Flatten the axes array for easier indexing

    for i, item in enumerate(items):
        # Filter data for the current item
        item_data = aggregated_data[aggregated_data['Item'] == item].sort_values('Month')

        # Calculate the Cumulative Density Function (CDF) for this item
        item_data['CDF'] = item_data['Value'].cumsum() / item_data['Value'].sum()

        # Plot the CDF for the current item in its respective subplot
        axes[i].plot(item_data['Month'], item_data['CDF'], marker='o', linestyle='-', label=f'Item {item}')
        axes[i].set_title(f'Item {item}')
        axes[i].set_xlabel('Month')
        axes[i].set_ylabel('CDF')
        axes[i].grid(True)

        # Set x-axis limits to start exactly at the first datapoint and end at the last
        axes[i].set_xlim(item_data['Month'].min(), item_data['Month'].max())

        # Ensure that each x-axis ticks only show months where data is present
        axes[i].set_xticks(item_data['Month'].values)

        # Rotate the x-axis labels for readability
        axes[i].tick_params(axis='x', rotation=45)

    # Remove empty subplots (if any)
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.suptitle('Cumulative Density Function of Test Utilisation Over Time by Item', fontsize=16)
    plt.show()

# Call the function to plot CDF for each 'Item' in separate subplots
plot_cdf_per_item(aggregated_data)


#### Self-organising maps

In [ ]:
import pandas as pd
from minisom import MiniSom
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

# Load the data
file_path = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/Phase 1 - Study 4 - Australian Genetic and Genomic Test Utilisation/Data Files/Processed_MBSGeneticsCount_20240325_175333.feather"
data = pd.read_feather(file_path)

# Check the number of unique items before any processing
unique_items_before = data['Item'].nunique()
print(f"Number of unique 'Item' categories before pivoting: {unique_items_before}")

# Group by Item and Month, summing 'Value' if necessary
grouped_data = data.groupby(['Item', 'Month']).agg({'Value': 'sum'}).reset_index()

# Pivot the table so that each row is an 'Item' and columns represent 'Month' values
pivoted_data = grouped_data.pivot(index='Item', columns='Month', values='Value').fillna(0)

# Check the number of unique items after pivoting
unique_items_after = pivoted_data.shape[0]
print(f"Number of unique 'Item' categories after pivoting: {unique_items_after}")

# If there's a difference, check for missing or excluded data
if unique_items_before != unique_items_after:
    missing_items = data[~data['Item'].isin(pivoted_data.index)]
    print("Missing or excluded items:")
    print(missing_items)

# Normalize the values
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(pivoted_data)

# SOME parameters
x, y = 10, 10  # Grid dimensions for the SOME

# Initialize and train the SOME
some = MiniSom(x, y, scaled_data.shape[1], sigma=1.0, learning_rate=0.5)
some.random_weights_init(scaled_data)
some.train_random(scaled_data, 100)  # Train for 100 iterations

# Get the winning nodes for each item
item_positions = np.array([some.winner(item) for item in scaled_data])

# Convert to a DataFrame for easier handling
cluster_assignments = pd.DataFrame({
    'Item': pivoted_data.index,
    'Cluster_X': item_positions[:, 0],
    'Cluster_Y': item_positions[:, 1]
})

# Save to CSV so you can inspect it and share if needed
output_file = "/mnt/c/Users/doughnut/OneDrive - The University of Melbourne/SOM_cluster_assignments.csv"
cluster_assignments.to_csv(output_file, index=False)

# Check the file output
print(cluster_assignments.head())
